# Qwen-27B Decision-Neutral Factor-to-Semantic Mediation

The response-side experiments establish a causal L49 physical-side semantic and its L55 conversion into an answer code.

[实验设置与运行说明](../../../docs/experiment_notes/10_exploratory_composition_and_gating/bd459ef089d07528.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Setup


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the decision-neutral mediation protocol
cell_stage('Freezing model, sites, supports, interventions, and decision gates')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
SEMANTIC_LAYER = 49
CAPTURE_LAYERS = (UPSTREAM_LAYER, SEMANTIC_LAYER)
UPSTREAM_SCOPE = 'options_tokens'
SEMANTIC_SCOPE = 'final_prompt_token'

SEED = 20260911
FACTORS = ('weight', 'distance')
FAMILIES_PER_FACTOR = 24
PILOT_PER_FACTOR = 4
CONFIRMATION_PER_FACTOR = FAMILIES_PER_FACTOR - PILOT_PER_FACTOR
BACKGROUNDS = ('b1', 'b2')
MAPPING_VARIANTS = ('forward', 'reverse')
INTERVENTIONS = (
    'additive_factor',
    'additive_factor_l49_block',
    'additive_factor_l49_random_block',
    'random_factor',
    'whole_donor',
    'natural_hybrid',
    'self_replay',
    'l49_oracle',
)
RECORDS_PER_FAMILY = len(BACKGROUNDS) * len(INTERVENTIONS)
EXPECTED_FAMILIES = len(FACTORS) * FAMILIES_PER_FACTOR
EXPECTED_RECORDS = EXPECTED_FAMILIES * RECORDS_PER_FAMILY
BOOTSTRAP_REPLICATES = 10000
ALPHA = 0.05
SEMANTIC_SESOI = 0.05
ANSWER_SESOI_LOGODDS = 0.10
MEDIATION_SESOI_LOGODDS = 0.05
ORACLE_SESOI_LOGODDS = 0.25
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 1e-3
IMAGE_TOKEN_ID_OVERRIDE = None

SEMANTIC_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_semantic_error_mediation_v1'
SEMANTIC_DECISION_PATH = (
    SEMANTIC_ROOT / 'analysis' / 'semantic_error_decision.json'
)
AXIS_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_symmetric_code_axis_v1'
SEMANTIC_TENSOR_PATH = AXIS_ROOT / 'direction_tensors.pt'
DIMENSION_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_response_subspace_dimension_transfer_v1'
)
DIMENSION_DECISION_PATH = (
    DIMENSION_ROOT / 'analysis' / 'dimension_transfer_decision.json'
)
DIMENSION_RUN_PATH = DIMENSION_ROOT / 'RUN_COMPLETE.json'

SHARED_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_decision_neutral_factor_semantic_mediation_v1'
)
MANIFEST_PATH = SHARED_ROOT / 'decision_neutral_manifest.jsonl'
MANIFEST_LOCK_PATH = SHARED_ROOT / 'manifest_lock.json'
PROTOCOL_LOCK_PATH = SHARED_ROOT / 'frozen_protocol.json'
SHARD_DIR = SHARED_ROOT / 'family_shards'
ANALYSIS_DIR = SHARED_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (SHARED_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

for required in (
    SEMANTIC_DECISION_PATH, SEMANTIC_TENSOR_PATH,
    DIMENSION_DECISION_PATH, DIMENSION_RUN_PATH,
):
    assert required.is_file(), f'Missing prerequisite: {required}'

semantic_decision = json.loads(
    SEMANTIC_DECISION_PATH.read_text(encoding='utf-8')
)
dimension_decision = json.loads(
    DIMENSION_DECISION_PATH.read_text(encoding='utf-8')
)
assert semantic_decision['primary_decision'] == (
    'bidirectional_semantic_error_control_with_code_mediation'
)
assert int(semantic_decision['semantic_layer']) == SEMANTIC_LAYER
assert dimension_decision['primary_decision'] == (
    'shared_core_with_codebook_specific_residual'
)

semantic_tensors = torch.load(
    SEMANTIC_TENSOR_PATH, map_location='cpu', weights_only=False
)
assert semantic_tensors['model_id'] == MODEL_ID
assert semantic_tensors['revision'] == MODEL_REVISION
SEMANTIC_DIRECTION = semantic_tensors['semantic_direction'].float()
if SEMANTIC_DIRECTION.ndim == 1:
    SEMANTIC_DIRECTION = SEMANTIC_DIRECTION.unsqueeze(0)
assert SEMANTIC_DIRECTION.ndim == 2 and SEMANTIC_DIRECTION.shape[0] == 1
SEMANTIC_STEERING_NORM = float(torch.linalg.vector_norm(SEMANTIC_DIRECTION))
assert SEMANTIC_STEERING_NORM > 0
SEMANTIC_UNIT = SEMANTIC_DIRECTION / SEMANTIC_STEERING_NORM

protocol_identity = {
    'schema_version': 'qwen27b-decision-neutral-factor-semantic-1.0',
    'analysis_status': 'frozen model-blind confirmatory factorial',
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'upstream_site': [UPSTREAM_LAYER, UPSTREAM_SCOPE],
    'semantic_site': [SEMANTIC_LAYER, SEMANTIC_SCOPE],
    'factors': list(FACTORS),
    'families_per_factor': FAMILIES_PER_FACTOR,
    'pilot_per_factor': PILOT_PER_FACTOR,
    'backgrounds': list(BACKGROUNDS),
    'mapping_variants': list(MAPPING_VARIANTS),
    'interventions': list(INTERVENTIONS),
    'donor_constraint': (
        'anchor, donor, and both targets share one outcome; both natural '
        'hybrids share the opposite outcome'
    ),
    'selection_uses_model_outputs': False,
    'primary_unit': 'factorial family averaged over two backgrounds',
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'semantic_sesoi': SEMANTIC_SESOI,
    'answer_sesoi_logodds': ANSWER_SESOI_LOGODDS,
    'mediation_sesoi_logodds': MEDIATION_SESOI_LOGODDS,
    'semantic_direction_sha256': sha256_file(SEMANTIC_TENSOR_PATH),
    'semantic_decision_sha256': sha256_file(SEMANTIC_DECISION_PATH),
    'dimension_decision_sha256': sha256_file(DIMENSION_DECISION_PATH),
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Model:', MODEL_ID, '@', MODEL_REVISION)
print('Frozen sites:', (UPSTREAM_LAYER, UPSTREAM_SCOPE),
      (SEMANTIC_LAYER, SEMANTIC_SCOPE))
print('Families / records:', EXPECTED_FAMILIES, EXPECTED_RECORDS)
print('L49 steering norm:', SEMANTIC_STEERING_NORM)
print('Output:', SHARED_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


## 2. Model-blind decision-neutral factor fanout


In [ ]:
%%cell_progress 6/17 Build the independent decision-neutral factorial manifest
cell_stage('Enumerating model-blind weight and distance fanout families')
import itertools
from collections import Counter, defaultdict

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + '\n')
    temporary.replace(path)

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(row['scene_id']): row for row in scene_records}

def single_shape(scene):
    if len(scene['stacks']) != 2:
        return None
    left = [stack for stack in scene['stacks'] if stack['side'] == 'left']
    right = [stack for stack in scene['stacks'] if stack['side'] == 'right']
    if len(left) != 1 or len(right) != 1:
        return None
    return (
        int(left[0]['units']), int(left[0]['distance']),
        int(right[0]['units']), int(right[0]['distance']),
    )

scene_by_shape = {}
for scene in scene_records:
    shape = single_shape(scene)
    if shape is not None:
        assert shape not in scene_by_shape
        scene_by_shape[shape] = str(scene['scene_id'])
assert len(scene_by_shape) == 900, len(scene_by_shape)

prior_scene_ids = set()
results_root = PROJECT_ROOT / 'results'
if results_root.is_dir():
    prior_paths = sorted(
        path for path in results_root.rglob('*manifest*.jsonl')
        if SHARED_ROOT not in path.parents
    )
    for path in tqdm(prior_paths, desc='Auditing prior scene support', unit='file'):
        try:
            rows = read_jsonl(path)
        except Exception:
            continue
        for row in rows:
            for key, value in row.items():
                if key.endswith('_scene_id') and isinstance(value, (str, int)):
                    prior_scene_ids.add(str(value))
                elif key.endswith('_scene_ids') and isinstance(value, (list, tuple)):
                    prior_scene_ids.update(str(item) for item in value)
            if isinstance(row.get('roles'), dict):
                for role in row['roles'].values():
                    if isinstance(role, dict) and 'scene_id' in role:
                        prior_scene_ids.add(str(role['scene_id']))

def shape_for(side, units, distance, fixed_units, fixed_distance):
    if side == 'left':
        return (units, distance, fixed_units, fixed_distance)
    return (fixed_units, fixed_distance, units, distance)

def role_record(side, units, distance, fixed_units, fixed_distance):
    shape = shape_for(side, units, distance, fixed_units, fixed_distance)
    scene_id = scene_by_shape.get(shape)
    if scene_id is None:
        return None
    scene = scene_by_id[scene_id]
    return {
        'scene_id': scene_id,
        'units': int(units),
        'distance': int(distance),
        'side_torque': int(scene['derived'][f'{side}_torque']),
        'net_torque': int(scene['derived']['net_torque']),
        'outcome': str(scene['derived']['outcome']),
    }

def decision_neutral(roles):
    base = roles['anchor']['outcome']
    hybrid = roles['hybrid_b1']['outcome']
    return (
        base in ('left', 'right')
        and hybrid in ('left', 'right')
        and base != hybrid
        and all(
            roles[name]['outcome'] == base
            for name in ('source', 'target_b1', 'target_b2')
        )
        and roles['hybrid_b2']['outcome'] == hybrid
    )

if MANIFEST_PATH.is_file() and MANIFEST_LOCK_PATH.is_file():
    factor_manifest = read_jsonl(MANIFEST_PATH)
    lock = json.loads(MANIFEST_LOCK_PATH.read_text(encoding='utf-8'))
    assert lock['identity']['selected_count'] == len(factor_manifest)
    candidates = None
    cell_stage('Using the existing locked manifest')
else:
    candidates = []
    for side in ('left', 'right'):
        for fixed_units in range(1, 6):
            for fixed_distance in range(1, 7):
                for w0, w1 in itertools.permutations(range(1, 6), 2):
                    for d0 in range(1, 7):
                        for d1, d2 in itertools.combinations(
                            [value for value in range(1, 7) if value != d0], 2
                        ):
                            values = {
                                'anchor': (w0, d0), 'source': (w1, d0),
                                'target_b1': (w0, d1), 'hybrid_b1': (w1, d1),
                                'target_b2': (w0, d2), 'hybrid_b2': (w1, d2),
                            }
                            roles = {
                                name: role_record(
                                    side, units, distance,
                                    fixed_units, fixed_distance,
                                )
                                for name, (units, distance) in values.items()
                            }
                            if None in roles.values() or not decision_neutral(roles):
                                continue
                            family_id = 'neutral_weight_' + stable_key((
                                side, fixed_units, fixed_distance,
                                w0, w1, d0, d1, d2,
                            ))[:18]
                            candidates.append({
                                'family_id': family_id,
                                'factor': 'weight',
                                'changed_side': side,
                                'fixed_opposite_units': fixed_units,
                                'fixed_opposite_distance': fixed_distance,
                                'factor_sign': int(w1 > w0) - int(w1 < w0),
                                'roles': roles,
                            })

                for d0, d1 in itertools.permutations(range(1, 7), 2):
                    for w0 in range(1, 6):
                        for w1, w2 in itertools.combinations(
                            [value for value in range(1, 6) if value != w0], 2
                        ):
                            values = {
                                'anchor': (w0, d0), 'source': (w0, d1),
                                'target_b1': (w1, d0), 'hybrid_b1': (w1, d1),
                                'target_b2': (w2, d0), 'hybrid_b2': (w2, d1),
                            }
                            roles = {
                                name: role_record(
                                    side, units, distance,
                                    fixed_units, fixed_distance,
                                )
                                for name, (units, distance) in values.items()
                            }
                            if None in roles.values() or not decision_neutral(roles):
                                continue
                            family_id = 'neutral_distance_' + stable_key((
                                side, fixed_units, fixed_distance,
                                d0, d1, w0, w1, w2,
                            ))[:18]
                            candidates.append({
                                'family_id': family_id,
                                'factor': 'distance',
                                'changed_side': side,
                                'fixed_opposite_units': fixed_units,
                                'fixed_opposite_distance': fixed_distance,
                                'factor_sign': int(d1 > d0) - int(d1 < d0),
                                'roles': roles,
                            })

    assert all(decision_neutral(row['roles']) for row in candidates)
    for row in candidates:
        row['prior_overlap_count'] = sum(
            role['scene_id'] in prior_scene_ids
            for role in row['roles'].values()
        )
        row['transition'] = (
            row['roles']['anchor']['outcome'] + '->'
            + row['roles']['hybrid_b1']['outcome']
        )

    selected = []
    used_scenes = set()
    for factor in FACTORS:
        groups = defaultdict(list)
        for row in candidates:
            if row['factor'] == factor:
                groups[(
                    row['changed_side'], row['factor_sign'], row['transition']
                )].append(row)
        for rows in groups.values():
            rows.sort(key=lambda row: (
                row['prior_overlap_count'], stable_key(row['family_id'])
            ))
        group_keys = sorted(groups, key=lambda value: stable_key(value))
        cursors = {key: 0 for key in group_keys}
        factor_selected = []
        while len(factor_selected) < FAMILIES_PER_FACTOR:
            changed = False
            for key in group_keys:
                rows = groups[key]
                while cursors[key] < len(rows):
                    row = rows[cursors[key]]
                    cursors[key] += 1
                    scene_ids = {
                        role['scene_id'] for role in row['roles'].values()
                    }
                    if scene_ids & used_scenes:
                        continue
                    row = dict(row)
                    row['split'] = (
                        'pilot' if len(factor_selected) < PILOT_PER_FACTOR
                        else 'confirmation'
                    )
                    row['counterbalance_phase'] = (
                        int(stable_key(f"mapping:{row['family_id']}")[:8], 16) % 2
                    )
                    factor_selected.append(row)
                    selected.append(row)
                    used_scenes.update(scene_ids)
                    changed = True
                    break
                if len(factor_selected) == FAMILIES_PER_FACTOR:
                    break
            if not changed:
                break
        assert len(factor_selected) == FAMILIES_PER_FACTOR, (
            factor, len(factor_selected), len(candidates)
        )

    factor_manifest = sorted(selected, key=lambda row: row['family_id'])
    write_jsonl_atomic(MANIFEST_PATH, factor_manifest)
    manifest_identity = {
        'schema_version': 'decision-neutral-factor-manifest-1.0',
        'selected_count': len(factor_manifest),
        'counts_by_factor': dict(Counter(
            row['factor'] for row in factor_manifest
        )),
        'counts_by_split': dict(Counter(
            row['split'] for row in factor_manifest
        )),
        'scene_disjoint': True,
        'selection_uses_model_outputs': False,
        'decision_neutral_donor': True,
        'two_background_fanout': True,
        'family_ids': [row['family_id'] for row in factor_manifest],
        'seed': SEED,
    }
    write_json_atomic(MANIFEST_LOCK_PATH, {
        'identity': manifest_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

assert len(factor_manifest) == EXPECTED_FAMILIES
assert Counter(row['factor'] for row in factor_manifest) == {
    factor: FAMILIES_PER_FACTOR for factor in FACTORS
}
all_selected_scenes = [
    role['scene_id']
    for row in factor_manifest for role in row['roles'].values()
]
assert len(all_selected_scenes) == len(set(all_selected_scenes))
assert all(decision_neutral(row['roles']) for row in factor_manifest)

manifest_frame = pd.DataFrame([{
    'family_id': row['family_id'],
    'factor': row['factor'],
    'split': row['split'],
    'changed_side': row['changed_side'],
    'factor_sign': row['factor_sign'],
    'transition': row['transition'],
    'counterbalance_phase': row['counterbalance_phase'],
    'prior_overlap_count': row['prior_overlap_count'],
} for row in factor_manifest])
manifest_frame.to_csv(ANALYSIS_DIR / 'manifest_summary.csv', index=False)
display(manifest_frame.groupby([
    'factor', 'split', 'transition', 'factor_sign'
]).size().rename('n').reset_index())
print('Candidate / selected families:',
      len(candidates) if candidates is not None else 'locked',
      len(factor_manifest))
print('Selected scenes:', len(all_selected_scenes))
print('Prior-overlap scenes:', sum(
    scene_id in prior_scene_ids for scene_id in all_selected_scenes
))

example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=SHARED_ROOT / '_unused_behavior_loader',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'),
    batch_size=1, generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(example_config))
outcome_by_scene = {
    str(example.scene_id): example
    for example in outcome_examples
    if example.variant == 'permutation_0'
}
assert len(outcome_by_scene) == len(scene_records)


In [ ]:
%%cell_progress 7/17 Load the pinned Qwen-27B decoder
cell_stage(f'Loading {MODEL_KEY} weights for recombination')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


## 3. L48 factor transport into the L49 physical semantic


In [ ]:
%%cell_progress 8/17 Define counterbalanced outcome prompts and exact token masks
cell_stage('Defining label-swapped outcome bundles')
import math
from dataclasses import replace as dataclass_replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = 'Which way will the balance scale tip?'
SEMANTIC_TEXT = {
    'left': 'left side', 'balanced': 'equal', 'right': 'right side',
}

def mapping_for(row, background_index):
    variant = MAPPING_VARIANTS[
        (int(row['counterbalance_phase']) + background_index) % 2
    ]
    semantics = (
        ('left', 'balanced', 'right')
        if variant == 'forward'
        else ('right', 'balanced', 'left')
    )
    return variant, tuple(zip(('A', 'B', 'C'), semantics))

def custom_prompt(mapping):
    options = '\n'.join(
        f'{label}. {SEMANTIC_TEXT[semantic]}'
        for label, semantic in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT,
        'Question: ' + QUESTION_TEXT,
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def make_example(row, role_name, mapping, background, mapping_variant):
    role = row['roles'][role_name]
    scene_id = str(role['scene_id'])
    semantics = tuple(value for _, value in mapping)
    labels = tuple(label for label, _ in mapping)
    outcome = str(role['outcome'])
    base = outcome_by_scene[scene_id]
    return dataclass_replace(
        base,
        example_id=(
            f"decision_neutral:{row['family_id']}:{background}:"
            f"{mapping_variant}:{role_name}"
        ),
        question_id=(
            f"decision_neutral_{row['family_id']}_{background}_"
            f"{mapping_variant}"
        ),
        question_kind='outcome',
        variant='decision_neutral_factor_semantic',
        prompt=custom_prompt(mapping),
        labels=labels,
        option_text=tuple(SEMANTIC_TEXT[value] for value in semantics),
        option_semantics=semantics,
        gold_label=labels[semantics.index(outcome)],
        gold_semantic=outcome,
    )

def make_bundle(row, background_index):
    background = BACKGROUNDS[background_index]
    target_role = f'target_{background}'
    hybrid_role = f'hybrid_{background}'
    mapping_variant, mapping = mapping_for(row, background_index)
    role_names = ('anchor', 'source', target_role, hybrid_role)
    examples = {
        role: make_example(
            row, role, mapping, background, mapping_variant
        )
        for role in role_names
    }
    target_semantic = row['roles'][target_role]['outcome']
    hybrid_semantic = row['roles'][hybrid_role]['outcome']
    assert target_semantic == row['roles']['source']['outcome']
    assert target_semantic != hybrid_semantic
    return {
        'background': background,
        'mapping_variant': mapping_variant,
        'mapping': mapping,
        'examples': examples,
        'anchor_role': 'anchor',
        'source_role': 'source',
        'target_role': target_role,
        'hybrid_role': hybrid_role,
        'target_semantic': target_semantic,
        'hybrid_semantic': hybrid_semantic,
    }

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {
        'examples': examples,
        'inputs': inputs,
        'token_by_label': token_by_label,
    }

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = (
            IMAGE_TOKEN_ID_OVERRIDE
            if isinstance(IMAGE_TOKEN_ID_OVERRIDE, (list, tuple))
            else [IMAGE_TOKEN_ID_OVERRIDE]
        )
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()],
        dtype=torch.bool,
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False,
        return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1, len(starts)
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    assert len(sections) == 5
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    masks = {
        'valid_tokens': attention,
        'image_tokens': image,
        'all_text_except_final': all_text,
        'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlaps = [
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        ]
        overlap, name = max(overlaps)
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    covered = torch.zeros_like(attention)
    for name in names:
        covered |= masks[name]
    masks['separator_chat_tokens'] = all_text & ~covered
    assert torch.equal(
        masks['options_tokens'] | masks['intro_tokens']
        | masks['rule_tokens'] | masks['question_tokens']
        | masks['answer_instruction_tokens']
        | masks['separator_chat_tokens'],
        all_text,
    )
    masks['tokens'] = tokens
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(reference['inputs'][key], current['inputs'][key]), (
                name, key
            )
        masks = discover_masks(current)
        for scope in (UPSTREAM_SCOPE, SEMANTIC_SCOPE):
            assert torch.equal(reference_masks[scope], masks[scope]), (
                name, scope
            )
    return reference_masks

print('Counterbalanced outcome prompts and masks ready.')


In [ ]:
%%cell_progress 9/17 Define L48 intervention and L49 mediation scorers
cell_stage('Defining hidden-state capture, semantic blocking, and answer scoring')

def score_record(prepared, logits):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    logprobs = torch.log_softmax(logits[0, -1, :].float(), dim=-1)
    scores = [
        float(logprobs[prepared['token_by_label'][label]].detach().cpu())
        for label in example.labels
    ]
    maximum = max(scores)
    weights = [math.exp(value - maximum) for value in scores]
    total = sum(weights)
    predicted_index = int(np.argmax(scores))
    return {
        'scene_id': str(example.scene_id),
        'example_id': str(example.example_id),
        'candidate_semantics': list(example.option_semantics),
        'candidate_logprobs': scores,
        'candidate_probabilities': [value / total for value in weights],
        'predicted_semantic': example.option_semantics[predicted_index],
        'gold_semantic': example.gold_semantic,
    }

def forward_capture(prepared):
    hidden_by_layer = {}
    handles = []
    for layer_index in CAPTURE_LAYERS:
        def hook(_module, _inputs, output, layer_index=layer_index):
            hidden_by_layer[layer_index] = (
                output_hidden_tensor(output)[0].detach().cpu().clone()
            )
        handles.append(DECODER_LAYERS[layer_index].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_record(prepared, output.logits), hidden_by_layer

def capture_named_examples(examples_by_name):
    scores = {}
    hidden = {}
    for name, example in examples_by_name.items():
        prepared = prepare_examples([example])
        scores[name], hidden[name] = forward_capture(prepared)
        del prepared
        torch.cuda.empty_cache()
    return scores, hidden

def matched_random_delta(delta, mask, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(delta.shape, generator=generator, dtype=torch.float32)
    selected = mask.bool()
    output = torch.zeros_like(noise)
    delta_selected = delta.float()[selected]
    noise_selected = noise[selected]
    delta_norm = torch.linalg.vector_norm(delta_selected)
    noise_norm = torch.linalg.vector_norm(noise_selected)
    assert float(delta_norm) > 0 and float(noise_norm) > 0
    output[selected] = noise_selected * (delta_norm / noise_norm)
    return output.to(delta.dtype)

def random_semantic_unit(key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    vector = torch.randn(
        SEMANTIC_UNIT.shape, generator=generator, dtype=torch.float32
    )
    vector = vector - (vector * SEMANTIC_UNIT).sum() * SEMANTIC_UNIT
    norm = torch.linalg.vector_norm(vector)
    assert float(norm) > 0
    return vector / norm

def score_intervention(example, clean_l49, condition):
    prepared = prepare_examples([example])
    masks = discover_masks(prepared)
    upstream_mask = masks[UPSTREAM_SCOPE]
    semantic_mask = masks[SEMANTIC_SCOPE]
    assert int(semantic_mask.sum()) == 1
    measurements = {}

    def upstream_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = condition['upstream_payload'].to(
            hidden.device, hidden.dtype
        )
        current = upstream_mask.to(hidden.device)
        modified = hidden.clone()
        mode = condition['upstream_mode']
        if mode == 'add':
            modified[0, current] = hidden[0, current] + payload[current]
        elif mode == 'replace':
            modified[0, current] = payload[current]
        else:
            raise ValueError(mode)
        measurements['upstream_payload_norm'] = float(
            torch.linalg.vector_norm(payload[current].float()).detach().cpu()
        )
        return replace_output_hidden(output, modified)

    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        current = semantic_mask.to(hidden.device)
        clean = clean_l49.to(hidden.device, torch.float32)
        before = hidden[0, current].float()
        delta = before - clean
        unit = SEMANTIC_UNIT.to(hidden.device, torch.float32)
        coordinate = (delta * unit).sum()
        desired_sign = 1.0 if condition['hybrid_semantic'] == 'left' else -1.0
        desired_coordinate = desired_sign * coordinate
        semantic_component = coordinate * unit
        semantic_component_norm = torch.linalg.vector_norm(semantic_component)
        mode = condition['semantic_mode']
        after = before
        random_component_norm = torch.tensor(0.0, device=hidden.device)
        if mode == 'block':
            after = before - semantic_component
        elif mode == 'random_block':
            random_unit = condition['random_semantic_unit'].to(
                hidden.device, torch.float32
            )
            random_component = random_unit * semantic_component_norm
            random_component_norm = torch.linalg.vector_norm(random_component)
            after = before - random_component
        elif mode == 'oracle':
            oracle = desired_sign * SEMANTIC_DIRECTION.to(
                hidden.device, torch.float32
            )
            after = before + oracle
        elif mode != 'none':
            raise ValueError(mode)
        after_delta = after - clean
        after_coordinate = desired_sign * (after_delta * unit).sum()
        measurements.update({
            'l49_delta_norm': float(
                torch.linalg.vector_norm(delta).detach().cpu()
            ),
            'l49_desired_coordinate': float(desired_coordinate.detach().cpu()),
            'l49_desired_coordinate_normalized': float(
                (desired_coordinate / SEMANTIC_STEERING_NORM).detach().cpu()
            ),
            'l49_after_coordinate_normalized': float(
                (after_coordinate / SEMANTIC_STEERING_NORM).detach().cpu()
            ),
            'semantic_component_norm': float(
                semantic_component_norm.detach().cpu()
            ),
            'random_component_norm': float(
                random_component_norm.detach().cpu()
            ),
        })
        if mode == 'none':
            return None
        modified = hidden.clone()
        modified[0, current] = after.to(hidden.dtype)
        return replace_output_hidden(output, modified)

    handles = [
        DECODER_LAYERS[UPSTREAM_LAYER].register_forward_hook(upstream_hook),
        DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(semantic_hook),
    ]
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_record(prepared, output.logits)
    record.update({
        key: value for key, value in condition.items()
        if key not in ('upstream_payload', 'random_semantic_unit')
    })
    record.update(measurements)
    record['upstream_patched_tokens'] = int(upstream_mask.sum())
    record['semantic_patched_tokens'] = (
        int(semantic_mask.sum())
        if condition['semantic_mode'] != 'none' else 0
    )
    return record

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

print('L48 intervention and L49 mediation scorers ready.')


In [ ]:
%%cell_progress 10/17 Run physics, alignment, semantic-axis, and parity smoke tests
cell_stage('Checking decision neutrality, exact masks, and replay parity')
smoke_row = next(row for row in factor_manifest if row['split'] == 'pilot')
smoke_bundle = make_bundle(smoke_row, 0)
assert decision_neutral(smoke_row['roles'])
assert smoke_bundle['target_semantic'] == (
    smoke_row['roles']['source']['outcome']
)
assert smoke_bundle['hybrid_semantic'] != smoke_bundle['target_semantic']

smoke_prepared = {
    name: prepare_examples([example])
    for name, example in smoke_bundle['examples'].items()
}
smoke_masks = assert_aligned(smoke_prepared)
assert bool(smoke_masks[UPSTREAM_SCOPE].any())
assert int(smoke_masks[SEMANTIC_SCOPE].sum()) == 1
smoke_scores, smoke_hidden = capture_named_examples(smoke_bundle['examples'])
target = smoke_bundle['target_role']
source = smoke_bundle['source_role']
anchor = smoke_bundle['anchor_role']
factor_delta = (
    smoke_hidden[source][UPSTREAM_LAYER]
    - smoke_hidden[anchor][UPSTREAM_LAYER]
)
clean_l49 = smoke_hidden[target][SEMANTIC_LAYER][
    smoke_masks[SEMANTIC_SCOPE]
]
smoke_random_unit = random_semantic_unit('smoke-l49')

common = {
    'family_id': smoke_row['family_id'],
    'factor': smoke_row['factor'],
    'background': smoke_bundle['background'],
    'mapping_variant': smoke_bundle['mapping_variant'],
    'target_semantic': smoke_bundle['target_semantic'],
    'hybrid_semantic': smoke_bundle['hybrid_semantic'],
    'random_semantic_unit': smoke_random_unit,
}
self_record = score_intervention(
    smoke_bundle['examples'][target], clean_l49, {
        **common, 'intervention': 'self_replay',
        'upstream_mode': 'replace',
        'upstream_payload': smoke_hidden[target][UPSTREAM_LAYER],
        'semantic_mode': 'none',
    }
)
zero_record = score_intervention(
    smoke_bundle['examples'][target], clean_l49, {
        **common, 'intervention': 'zero_add',
        'upstream_mode': 'add',
        'upstream_payload': torch.zeros_like(factor_delta),
        'semantic_mode': 'none',
    }
)
factor_record = score_intervention(
    smoke_bundle['examples'][target], clean_l49, {
        **common, 'intervention': 'factor_random_block_smoke',
        'upstream_mode': 'add',
        'upstream_payload': factor_delta,
        'semantic_mode': 'random_block',
    }
)
self_error = float(np.max(np.abs(
    centered_scores(self_record) - centered_scores(smoke_scores[target])
)))
zero_error = float(np.max(np.abs(
    centered_scores(zero_record) - centered_scores(smoke_scores[target])
)))
assert self_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, self_error
assert zero_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, zero_error
semantic_norm = factor_record['semantic_component_norm']
random_norm = factor_record['random_component_norm']
if semantic_norm > 1e-10:
    relative_error = abs(random_norm - semantic_norm) / semantic_norm
    assert relative_error < NORM_RELATIVE_TOLERANCE, relative_error
else:
    relative_error = 0.0
assert self_record['upstream_patched_tokens'] == int(
    smoke_masks[UPSTREAM_SCOPE].sum()
)
print('Centered self / zero parity:', self_error, zero_error)
print('Random-block norm relative error:', relative_error)
print('Smoke outcomes donor/target -> hybrid:',
      smoke_bundle['target_semantic'], '->', smoke_bundle['hybrid_semantic'])
print('Patched token counts:',
      self_record['upstream_patched_tokens'],
      factor_record['semantic_patched_tokens'])
del smoke_prepared, smoke_hidden, factor_delta
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 11/17 Capture or resume decision-neutral factor interventions
cell_stage('Locking capture identity and checking resumable family shards')

capture_identity = {
    'schema_version': 'decision-neutral-factor-semantic-capture-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'semantic_tensor_sha256': sha256_file(SEMANTIC_TENSOR_PATH),
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'upstream_layer': UPSTREAM_LAYER,
    'semantic_layer': SEMANTIC_LAYER,
    'interventions': list(INTERVENTIONS),
    'records_per_family': RECORDS_PER_FAMILY,
    'expected_records': EXPECTED_RECORDS,
    'layer_path': LAYER_PATH,
}
capture_config_path = SHARED_ROOT / 'capture_config.json'
if capture_config_path.is_file():
    previous = json.loads(capture_config_path.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity, (
        'Existing capture differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(capture_config_path, {
        'identity': capture_identity,
        'adapter_metadata': adapter.metadata,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    })
CAPTURE_SHA256 = sha256_file(capture_config_path)

completed_ids = set()
completed_records = 0
for path in sorted(SHARD_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    assert len(payload['records']) == RECORDS_PER_FAMILY
    completed_ids.add(str(payload['family']['family_id']))
    completed_records += len(payload['records'])
print('Completed families:', len(completed_ids), '/', EXPECTED_FAMILIES)

progress = tqdm(
    total=EXPECTED_RECORDS, initial=completed_records,
    desc='Decision-neutral factor mediation', unit='condition',
    dynamic_ncols=True,
)
for row in factor_manifest:
    family_id = str(row['family_id'])
    if family_id in completed_ids:
        continue
    family_cells = []
    family_records = []
    for background_index, background in enumerate(BACKGROUNDS):
        bundle = make_bundle(row, background_index)
        prepared = {
            name: prepare_examples([example])
            for name, example in bundle['examples'].items()
        }
        masks = assert_aligned(prepared)
        scores, hidden = capture_named_examples(bundle['examples'])
        anchor = bundle['anchor_role']
        source = bundle['source_role']
        target = bundle['target_role']
        hybrid = bundle['hybrid_role']
        factor_delta = (
            hidden[source][UPSTREAM_LAYER]
            - hidden[anchor][UPSTREAM_LAYER]
        )
        random_delta = matched_random_delta(
            factor_delta, masks[UPSTREAM_SCOPE],
            f"upstream:{family_id}:{background}"
        )
        random_unit = random_semantic_unit(
            f"semantic:{family_id}:{background}"
        )
        clean_l49 = hidden[target][SEMANTIC_LAYER][masks[SEMANTIC_SCOPE]]
        condition_specs = {
            'additive_factor': ('add', factor_delta, 'none'),
            'additive_factor_l49_block': ('add', factor_delta, 'block'),
            'additive_factor_l49_random_block': (
                'add', factor_delta, 'random_block'
            ),
            'random_factor': ('add', random_delta, 'none'),
            'whole_donor': (
                'replace', hidden[source][UPSTREAM_LAYER], 'none'
            ),
            'natural_hybrid': (
                'replace', hidden[hybrid][UPSTREAM_LAYER], 'none'
            ),
            'self_replay': (
                'replace', hidden[target][UPSTREAM_LAYER], 'none'
            ),
            'l49_oracle': (
                'replace', hidden[target][UPSTREAM_LAYER], 'oracle'
            ),
        }
        records = []
        for intervention in INTERVENTIONS:
            upstream_mode, upstream_payload, semantic_mode = (
                condition_specs[intervention]
            )
            record = score_intervention(
                bundle['examples'][target], clean_l49, {
                    'family_id': family_id,
                    'factor': row['factor'],
                    'split': row['split'],
                    'background': background,
                    'mapping_variant': bundle['mapping_variant'],
                    'transition': row['transition'],
                    'factor_sign': row['factor_sign'],
                    'target_semantic': bundle['target_semantic'],
                    'hybrid_semantic': bundle['hybrid_semantic'],
                    'intervention': intervention,
                    'upstream_mode': upstream_mode,
                    'upstream_payload': upstream_payload,
                    'semantic_mode': semantic_mode,
                    'random_semantic_unit': random_unit,
                }
            )
            records.append(record)
            family_records.append(record)
            progress.update(1)
            progress.set_postfix_str(
                f"{row['factor']} | {family_id} | {background} | {intervention}"
            )
        family_cells.append({
            'background': background,
            'mapping_variant': bundle['mapping_variant'],
            'target_semantic': bundle['target_semantic'],
            'hybrid_semantic': bundle['hybrid_semantic'],
            'baseline_scores': scores,
            'records': records,
        })
        del prepared, hidden, factor_delta, random_delta, clean_l49
        torch.cuda.empty_cache()
    assert len(family_records) == RECORDS_PER_FAMILY
    write_json_atomic(SHARD_DIR / f'{family_id}.json', {
        'schema_version': 'decision-neutral-factor-family-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'family': row,
        'cells': family_cells,
        'records': family_records,
    })
progress.close()

all_shards = sorted(SHARD_DIR.glob('*.json'))
assert len(all_shards) == EXPECTED_FAMILIES
observed_records = 0
for path in all_shards:
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    assert len(payload['records']) == RECORDS_PER_FAMILY
    observed_records += len(payload['records'])
assert observed_records == EXPECTED_RECORDS
write_json_atomic(SHARED_ROOT / 'capture_complete.json', {
    'families': EXPECTED_FAMILIES,
    'records': EXPECTED_RECORDS,
    'capture_identity_sha256': CAPTURE_SHA256,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
})
print('Capture complete:', EXPECTED_RECORDS, 'records')


## 4. Frozen recombination and serial-mediation decision


In [ ]:
%%cell_progress 12/17 Release GPU and assemble serial-mediation estimands
cell_stage('Releasing model memory and calculating matched causal contrasts')
import gc

del adapter
gc.collect()
torch.cuda.empty_cache()

def semantic_logprob(record, semantic):
    semantics = list(record['candidate_semantics'])
    return float(record['candidate_logprobs'][semantics.index(semantic)])

def lookup(records, intervention):
    matches = [
        record for record in records
        if record['intervention'] == intervention
    ]
    assert len(matches) == 1
    return matches[0]

cell_rows = []
for path in tqdm(
    sorted(SHARD_DIR.glob('*.json')),
    desc='Assembling family estimands', unit='family',
):
    payload = json.loads(path.read_text(encoding='utf-8'))
    family = payload['family']
    for cell in payload['cells']:
        target = cell['target_semantic']
        hybrid = cell['hybrid_semantic']
        clean = cell['baseline_scores'][
            f"target_{cell['background']}"
        ]
        clean_margin = (
            semantic_logprob(clean, hybrid)
            - semantic_logprob(clean, target)
        )
        records = {
            mode: lookup(cell['records'], mode)
            for mode in INTERVENTIONS
        }
        effects = {}
        for mode, record in records.items():
            margin = (
                semantic_logprob(record, hybrid)
                - semantic_logprob(record, target)
            )
            effects[mode] = margin - clean_margin
        factor_effect = effects['additive_factor']
        semantic_attenuation = (
            factor_effect - effects['additive_factor_l49_block']
        )
        random_attenuation = (
            factor_effect
            - effects['additive_factor_l49_random_block']
        )
        self_error = float(np.max(np.abs(
            centered_scores(records['self_replay']) - centered_scores(clean)
        )))
        semantic_norm = records[
            'additive_factor_l49_random_block'
        ]['semantic_component_norm']
        random_norm = records[
            'additive_factor_l49_random_block'
        ]['random_component_norm']
        norm_error = (
            abs(random_norm - semantic_norm) / semantic_norm
            if semantic_norm > 1e-10 else 0.0
        )
        cell_rows.append({
            'family_id': family['family_id'],
            'factor': family['factor'],
            'split': family['split'],
            'background': cell['background'],
            'mapping_variant': cell['mapping_variant'],
            'transition': family['transition'],
            'factor_sign': family['factor_sign'],
            'prior_overlap_count': family['prior_overlap_count'],
            'clean_hybrid_target_margin': clean_margin,
            'l49_factor_shift': records[
                'additive_factor'
            ]['l49_desired_coordinate_normalized'],
            'l49_random_shift': records[
                'random_factor'
            ]['l49_desired_coordinate_normalized'],
            'l49_factor_vs_random': (
                records['additive_factor'][
                    'l49_desired_coordinate_normalized'
                ]
                - records['random_factor'][
                    'l49_desired_coordinate_normalized'
                ]
            ),
            'answer_factor_effect': factor_effect,
            'answer_random_effect': effects['random_factor'],
            'answer_donor_effect': effects['whole_donor'],
            'answer_natural_hybrid_effect': effects['natural_hybrid'],
            'answer_oracle_effect': effects['l49_oracle'],
            'answer_factor_vs_random': (
                factor_effect - effects['random_factor']
            ),
            'answer_factor_vs_donor': (
                factor_effect - effects['whole_donor']
            ),
            'semantic_mediation_attenuation': semantic_attenuation,
            'random_block_attenuation': random_attenuation,
            'semantic_mediation_specificity': (
                semantic_attenuation - random_attenuation
            ),
            'self_replay_error': self_error,
            'random_block_norm_error': norm_error,
        })

cell_frame = pd.DataFrame(cell_rows)
assert len(cell_frame) == EXPECTED_FAMILIES * len(BACKGROUNDS)
cell_frame.to_csv(ANALYSIS_DIR / 'cell_estimands.csv', index=False)

metric_columns = [
    'l49_factor_shift', 'l49_random_shift', 'l49_factor_vs_random',
    'answer_factor_effect', 'answer_random_effect', 'answer_donor_effect',
    'answer_natural_hybrid_effect', 'answer_oracle_effect',
    'answer_factor_vs_random', 'answer_factor_vs_donor',
    'semantic_mediation_attenuation', 'random_block_attenuation',
    'semantic_mediation_specificity',
]
family_rows = []
for family_id, current in cell_frame.groupby('family_id'):
    assert len(current) == len(BACKGROUNDS)
    row = {
        'family_id': family_id,
        'factor': current['factor'].iloc[0],
        'split': current['split'].iloc[0],
        'transition': current['transition'].iloc[0],
        'fanout_minimum_answer_effect': float(
            current['answer_factor_effect'].min()
        ),
        'fanout_minimum_l49_shift': float(
            current['l49_factor_shift'].min()
        ),
        'maximum_self_replay_error': float(
            current['self_replay_error'].max()
        ),
        'maximum_random_block_norm_error': float(
            current['random_block_norm_error'].max()
        ),
    }
    row.update({
        metric: float(current[metric].mean())
        for metric in metric_columns
    })
    family_rows.append(row)
family_frame = pd.DataFrame(family_rows)
assert len(family_frame) == EXPECTED_FAMILIES
family_frame.to_csv(ANALYSIS_DIR / 'family_estimands.csv', index=False)
display(family_frame.groupby(['factor', 'split'])[
    metric_columns + ['fanout_minimum_answer_effect']
].mean().round(4))


In [ ]:
%%cell_progress 13/17 Run frozen factor-to-semantic and mediation tests
cell_stage('Bootstrapping confirmation families without behavior filtering')

def bootstrap_mean(values, key, replicates=BOOTSTRAP_REPLICATES):
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    assert len(array) > 1, (key, len(array))
    generator = np.random.default_rng(
        int(stable_key(f'bootstrap:{key}')[:16], 16) % (2**63 - 1)
    )
    draws = generator.choice(
        array, size=(replicates, len(array)), replace=True
    ).mean(axis=1)
    return {
        'n': int(len(array)),
        'mean': float(array.mean()),
        'ci_low': float(np.quantile(draws, ALPHA / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA / 2)),
    }

confirmation = family_frame[family_frame['split'] == 'confirmation'].copy()
assert Counter(confirmation['factor']) == {
    factor: CONFIRMATION_PER_FACTOR for factor in FACTORS
}

PRIMARY_METRICS = (
    'l49_factor_shift',
    'l49_factor_vs_random',
    'answer_factor_effect',
    'answer_factor_vs_random',
    'answer_factor_vs_donor',
    'fanout_minimum_answer_effect',
    'semantic_mediation_attenuation',
    'semantic_mediation_specificity',
    'answer_natural_hybrid_effect',
    'answer_oracle_effect',
)
test_rows = []
factor_decisions = {}
for factor in FACTORS:
    current = confirmation[confirmation['factor'] == factor]
    summaries = {}
    for metric in PRIMARY_METRICS:
        summary = bootstrap_mean(
            current[metric], f'{factor}:{metric}'
        )
        summaries[metric] = summary
        test_rows.append({
            'factor': factor, 'metric': metric, **summary,
        })
    representation_passed = (
        summaries['l49_factor_shift']['ci_low'] > SEMANTIC_SESOI
        and summaries['l49_factor_vs_random']['ci_low'] > 0
    )
    behavior_passed = (
        summaries['answer_factor_effect']['ci_low'] > ANSWER_SESOI_LOGODDS
        and summaries['answer_factor_vs_random']['ci_low'] > 0
        and summaries['answer_factor_vs_donor']['ci_low'] > 0
        and summaries['fanout_minimum_answer_effect']['ci_low'] > 0
    )
    mediation_passed = (
        summaries['semantic_mediation_attenuation']['ci_low']
        > MEDIATION_SESOI_LOGODDS
        and summaries['semantic_mediation_specificity']['ci_low'] > 0
    )
    controls_passed = (
        summaries['answer_natural_hybrid_effect']['ci_low'] > 0
        and summaries['answer_oracle_effect']['ci_low']
        > ORACLE_SESOI_LOGODDS
    )
    factor_decisions[factor] = {
        'passed': bool(
            representation_passed and behavior_passed
            and mediation_passed and controls_passed
        ),
        'representation_passed': bool(representation_passed),
        'behavior_passed': bool(behavior_passed),
        'mediation_passed': bool(mediation_passed),
        'controls_passed': bool(controls_passed),
        'summaries': summaries,
    }

passed_factors = [
    factor for factor, value in factor_decisions.items() if value['passed']
]
if len(passed_factors) == 2:
    PRIMARY_DECISION = 'both_factors_recombine_through_l49'
elif len(passed_factors) == 1:
    PRIMARY_DECISION = 'one_factor_recombines_through_l49'
elif any(
    value['representation_passed'] for value in factor_decisions.values()
):
    PRIMARY_DECISION = 'factor_signal_reaches_l49_without_full_mediation'
else:
    PRIMARY_DECISION = 'no_decision_neutral_factor_recombination_at_frozen_site'

tests_frame = pd.DataFrame(test_rows)
tests_frame.to_csv(ANALYSIS_DIR / 'confirmatory_tests.csv', index=False)
decision_rows = pd.DataFrame([{
    'factor': factor,
    **{key: value for key, value in status.items() if key != 'summaries'},
} for factor, status in factor_decisions.items()])
decision_rows.to_csv(ANALYSIS_DIR / 'factor_decisions.csv', index=False)
print('Frozen primary decision:', PRIMARY_DECISION)
display(tests_frame)
display(decision_rows)


In [ ]:
%%cell_progress 14/17 Audit physics, support, mappings, norms, and replay
cell_stage('Verifying decision-neutral donors and numerical integrity')

physics_violations = []
mapping_counts = Counter()
for row in factor_manifest:
    roles = row['roles']
    if not decision_neutral(roles):
        physics_violations.append(row['family_id'])
    for background_index, background in enumerate(BACKGROUNDS):
        variant, _ = mapping_for(row, background_index)
        mapping_counts[(row['factor'], row['split'], variant)] += 1

selected_scene_ids = [
    role['scene_id']
    for row in factor_manifest for role in row['roles'].values()
]
scene_disjoint = len(selected_scene_ids) == len(set(selected_scene_ids))
max_self_error = float(cell_frame['self_replay_error'].max())
max_norm_error = float(cell_frame['random_block_norm_error'].max())
nonfinite_count = int((~np.isfinite(
    family_frame.select_dtypes(include=[np.number]).to_numpy()
)).sum())

assert not physics_violations, physics_violations[:3]
assert scene_disjoint
assert max_self_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, max_self_error
assert max_norm_error < NORM_RELATIVE_TOLERANCE, max_norm_error
assert nonfinite_count == 0
assert all(
    mapping_counts[(factor, split, 'forward')]
    == mapping_counts[(factor, split, 'reverse')]
    for factor in FACTORS for split in ('pilot', 'confirmation')
)

audit_payload = {
    'schema_version': 'decision-neutral-factor-semantic-audit-1.0',
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'capture_sha256': CAPTURE_SHA256,
    'semantic_tensor_sha256': sha256_file(SEMANTIC_TENSOR_PATH),
    'families': len(factor_manifest),
    'records': EXPECTED_RECORDS,
    'physics_violations': len(physics_violations),
    'all_donors_decision_neutral': True,
    'all_targets_match_donor_outcome': True,
    'all_hybrids_flip_outcome': True,
    'two_background_fanout_per_family': True,
    'scene_disjoint': scene_disjoint,
    'selection_uses_model_outputs': False,
    'mapping_counts': {
        '|'.join(key): value for key, value in sorted(mapping_counts.items())
    },
    'maximum_self_replay_error': max_self_error,
    'self_replay_tolerance': SMOKE_CENTERED_LOGPROB_TOLERANCE,
    'maximum_random_block_norm_error': max_norm_error,
    'norm_tolerance': NORM_RELATIVE_TOLERANCE,
    'nonfinite_estimands': nonfinite_count,
    'prior_overlap_scenes': int(sum(
        scene_id in prior_scene_ids for scene_id in selected_scene_ids
    )),
}
write_json_atomic(ANALYSIS_DIR / 'robustness_audit.json', audit_payload)
display(pd.DataFrame([audit_payload]))


In [ ]:
%%cell_progress 15/17 Save the paper-facing factor-to-semantic figure
cell_stage('Rendering representation, behavior, mediation, and family panels')
import matplotlib.pyplot as plt

summary_lookup = {
    (row['factor'], row['metric']): row
    for _, row in tests_frame.iterrows()
}
colors = {'weight': '#D97706', 'distance': '#2563EB'}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def draw_metric_panel(axis, metrics, labels, title):
    x = np.arange(len(metrics))
    width = 0.34
    for offset, factor in enumerate(FACTORS):
        rows = [summary_lookup[(factor, metric)] for metric in metrics]
        means = np.asarray([row['mean'] for row in rows], dtype=float)
        lows = np.asarray([row['ci_low'] for row in rows], dtype=float)
        highs = np.asarray([row['ci_high'] for row in rows], dtype=float)
        positions = x + (offset - 0.5) * width
        axis.bar(
            positions, means, width,
            color=colors[factor], label=factor.capitalize(), alpha=0.88,
        )
        axis.errorbar(
            positions, means,
            yerr=np.vstack([means - lows, highs - means]),
            fmt='none', ecolor='black', capsize=3, linewidth=1,
        )
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_xticks(x, labels)
    axis.set_title(title)

draw_metric_panel(
    axes[0, 0],
    ['l49_factor_shift', 'l49_factor_vs_random'],
    ['Factor shift', 'vs random'],
    'A. L49 physical-semantic shift',
)
axes[0, 0].set_ylabel('Normalized semantic coordinate')

draw_metric_panel(
    axes[0, 1],
    ['answer_factor_effect', 'answer_factor_vs_random',
     'answer_factor_vs_donor', 'fanout_minimum_answer_effect'],
    ['Factor', 'vs random', 'vs donor', 'fanout min'],
    'B. Hybrid-answer causal effect',
)
axes[0, 1].set_ylabel('Hybrid−target log-odds shift')

draw_metric_panel(
    axes[1, 0],
    ['semantic_mediation_attenuation',
     'semantic_mediation_specificity'],
    ['L49 block attenuation', 'vs random block'],
    'C. Serial mediation through L49',
)
axes[1, 0].set_ylabel('Attenuation in log odds')

confirmation_plot = family_frame[
    family_frame['split'] == 'confirmation'
]
for factor in FACTORS:
    current = confirmation_plot[confirmation_plot['factor'] == factor]
    axes[1, 1].scatter(
        current['l49_factor_shift'], current['answer_factor_effect'],
        s=45, alpha=0.75, color=colors[factor], label=factor.capitalize(),
    )
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_xlabel('L49 normalized semantic shift')
axes[1, 1].set_ylabel('Hybrid−target answer shift')
axes[1, 1].set_title('D. Confirmation families')
axes[1, 1].legend(frameon=False)

for axis in axes.flat:
    axis.spines[['top', 'right']].set_visible(False)
axes[0, 0].legend(frameon=False)
fig.suptitle(
    'Decision-neutral factor transport into the L49 physical semantic',
    fontsize=15,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
figure_path = FIGURE_DIR / 'factor_to_semantic_mediation.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Figure:', figure_path)


In [ ]:
%%cell_progress 16/17 Write the frozen mechanism decision and report
cell_stage('Writing the machine-readable and paper-facing result cards')

if PRIMARY_DECISION == 'both_factors_recombine_through_l49':
    interpretation = (
        'Decision-neutral weight and distance deltas both form the new '
        'background-appropriate L49 physical semantic, affect the answer '
        'across two backgrounds, and lose that effect under a specific L49 '
        'semantic block. This supports independent factor composition into '
        'the response semantic at the frozen Qwen-27B site.'
    )
elif PRIMARY_DECISION == 'one_factor_recombines_through_l49':
    interpretation = (
        'Only one factor passes the full representation, fanout, behavior, '
        'and L49 mediation gate. Qwen-27B therefore shows asymmetric rather '
        'than complete weight–distance composition at the frozen site.'
    )
elif PRIMARY_DECISION == 'factor_signal_reaches_l49_without_full_mediation':
    interpretation = (
        'A decision-neutral factor signal reaches the L49 semantic readout, '
        'but it does not pass the full fanout and causal-mediation gate. '
        'Describe this as partial upstream influence, not an independently '
        'composed physical factor program.'
    )
else:
    interpretation = (
        'Neither decision-neutral factor passes the frozen L48-to-L49 '
        'recombination gate. Together with the positive L49-to-answer '
        'mechanism, this localizes the strongest supported computation to '
        'response-semantic control while limiting an upstream independent '
        'weight–distance composition claim.'
    )

decision_payload = {
    'schema_version': 'decision-neutral-factor-semantic-decision-1.0',
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'primary_decision': PRIMARY_DECISION,
    'factor_decisions': factor_decisions,
    'interpretation': interpretation,
    'audit': audit_payload,
    'limitations': [
        'The causal claim is local to Qwen-27B L48 options tokens and the '
        'frozen L49 final-prompt-token semantic.',
        'Failure at this site does not prove that factor composition is '
        'absent at every layer, token, or nonlinear subspace.',
        'The L49 block removes the validated one-dimensional physical-side '
        'semantic component; additional semantic dimensions may remain.',
        'The scenes use single stacks per side and two fanout backgrounds.',
        'No family is filtered by model correctness or confidence.',
    ],
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(ANALYSIS_DIR / 'factor_semantic_decision.json', decision_payload)

report_lines = [
    '# Decision-Neutral Factor-to-Semantic Mediation', '',
    f"- Model: `{MODEL_ID}` @ `{MODEL_REVISION}`",
    f"- Frozen decision: `{PRIMARY_DECISION}`",
    f"- Families: `{EXPECTED_FAMILIES}` "
    f"(`{2 * CONFIRMATION_PER_FACTOR}` confirmation)",
    f"- Records: `{EXPECTED_RECORDS}`", '',
    '## Interpretation', '', interpretation, '',
    '## Factor gates', '',
]
for factor in FACTORS:
    status = factor_decisions[factor]
    report_lines.extend([
        f"### {factor.capitalize()}", '',
        f"- Full gate: `{status['passed']}`",
        f"- L49 representation: `{status['representation_passed']}`",
        f"- Answer and fanout: `{status['behavior_passed']}`",
        f"- L49 mediation: `{status['mediation_passed']}`",
        f"- Positive controls: `{status['controls_passed']}`", '',
    ])
report_lines.extend([
    '## Claim boundary', '',
    'This experiment asks whether a factor state that contains no useful '
    'donor answer can be recombined with a new background before the validated '
    'L49 response semantic. It does not search every possible internal site.',
])
(ANALYSIS_DIR / 'REPORT.md').write_text(
    '\n'.join(report_lines) + '\n', encoding='utf-8'
)
display(pd.DataFrame([{
    'model_id': MODEL_ID,
    'primary_decision': PRIMARY_DECISION,
    'weight_passed': factor_decisions['weight']['passed'],
    'distance_passed': factor_decisions['distance']['passed'],
}]))
print(interpretation)


In [ ]:
%%cell_progress 17/17 Save paper-facing tables and completion manifest
cell_stage('Writing final artifact hashes and completion sentinel')

for frame, filename in (
    (manifest_frame, 'paper_manifest.csv'),
    (cell_frame, 'paper_cell_estimands.csv'),
    (family_frame, 'paper_family_estimands.csv'),
    (tests_frame, 'paper_confirmatory_tests.csv'),
    (decision_rows, 'paper_factor_decisions.csv'),
):
    frame.to_csv(ANALYSIS_DIR / filename, index=False)

artifact_paths = [
    PROTOCOL_LOCK_PATH, MANIFEST_PATH, MANIFEST_LOCK_PATH,
    SHARED_ROOT / 'capture_config.json',
    SHARED_ROOT / 'capture_complete.json',
    ANALYSIS_DIR / 'cell_estimands.csv',
    ANALYSIS_DIR / 'family_estimands.csv',
    ANALYSIS_DIR / 'confirmatory_tests.csv',
    ANALYSIS_DIR / 'factor_semantic_decision.json',
    ANALYSIS_DIR / 'robustness_audit.json',
    ANALYSIS_DIR / 'REPORT.md',
    FIGURE_DIR / 'factor_to_semantic_mediation.png',
]
for path in artifact_paths:
    assert Path(path).is_file(), path

run_manifest = {
    'schema_version': 'decision-neutral-factor-semantic-run-1.0',
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'primary_decision': PRIMARY_DECISION,
    'families': EXPECTED_FAMILIES,
    'records': EXPECTED_RECORDS,
    'capture_identity_sha256': CAPTURE_SHA256,
    'artifact_sha256': {
        str(Path(path).relative_to(SHARED_ROOT)): sha256_file(path)
        for path in artifact_paths
    },
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(SHARED_ROOT / 'RUN_COMPLETE.json', run_manifest)
display(decision_payload)
print('Run complete:', SHARED_ROOT / 'RUN_COMPLETE.json')
print('Decision:', PRIMARY_DECISION)
